# Module 16: MLOps Pipeline - CIFAR-10 | Local Mac Setup
### Mentor: Nethaji Nirmal | Applied Deep Learning to Production
---
**Running 100% locally on your Mac - no Colab, no ngrok needed!**

| Service | URL |
|---|---|
| MLflow Dashboard | http://localhost:`MLFLOW_PORT` (auto-detected in Cell 3) |
| FastAPI Swagger | http://localhost:8000/docs |

> Just run cells top-to-bottom. Virtual env + packages are all handled inside the notebook.

> **⚠️ macOS Note (Port 5000 conflict):**  
> On macOS Monterey and later, **port 5000 is occupied by AirPlay Receiver** (ControlCenter).  
> This notebook **auto-detects** a free port — it tries `5000` first, and falls back to `5050` if needed.  
> The chosen port is stored in the `MLFLOW_PORT` variable and used everywhere automatically.  
> *To manually free port 5000: System Settings → General → AirDrop & Handoff → AirPlay Receiver → Off.*


## Step 0: Create Virtual Environment & Install Packages
> Run ONCE with your base Python. After this cell:
> **Kernel -> Change Kernel -> Python (mlops_env)** then re-run from Cell 2.

In [1]:
# CELL 1 - Create virtual environment and install all dependencies (Mac)
import subprocess, sys, os

PROJECT_DIR = os.path.abspath('')
VENV_DIR    = os.path.join(PROJECT_DIR, 'mlops_env')
PIP         = os.path.join(VENV_DIR, 'bin', 'pip')
PYTHON      = os.path.join(VENV_DIR, 'bin', 'python')

os.makedirs(PROJECT_DIR, exist_ok=True)
print(f'Project folder: {PROJECT_DIR}')

if not os.path.exists(VENV_DIR):
    print('Creating virtual environment mlops_env ...')
    subprocess.run([sys.executable, '-m', 'venv', VENV_DIR], check=True)
    print('Virtual environment created.')
else:
    print('Virtual environment already exists, skipping.')

subprocess.run([PIP, 'install', '--upgrade', 'pip', '-q'], check=True)

PACKAGES = [
    'torch', 'torchvision', 'torchaudio',
    'fastapi', 'uvicorn[standard]', 'mlflow',
    'pydantic', 'nest-asyncio', 'requests',
    'ipykernel', 'ipywidgets', 'matplotlib'
]
print('Installing packages (2-5 min first time)...')
subprocess.run([PIP, 'install'] + PACKAGES + ['-q'], check=True)
print('All packages installed!')

subprocess.run([PYTHON, '-m', 'ipykernel', 'install',
                '--user', '--name', 'mlops_env',
                '--display-name', 'Python (mlops_env)'], check=True)

print('')
print('=== SETUP COMPLETE ===')
print('ACTION: Kernel -> Change Kernel -> Python (mlops_env)')
print('Then re-run all cells from Cell 2 onwards.')

Project folder: /Users/nirmalg/Desktop/guvi/mlops
Creating virtual environment mlops_env ...
Virtual environment created.
Installing packages (2-5 min first time)...
All packages installed!
Installed kernelspec mlops_env in /Users/nirmalg/Library/Jupyter/kernels/mlops_env

=== SETUP COMPLETE ===
ACTION: Kernel -> Change Kernel -> Python (mlops_env)
Then re-run all cells from Cell 2 onwards.


## Step 1: Verify Environment

In [ ]:
# CELL 2 - Verify all imports and detect hardware
import sys, torch, mlflow, fastapi
print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'MLflow  : {mlflow.__version__}')
print(f'FastAPI : {fastapi.__version__}')

if torch.backends.mps.is_available():
    print('Apple Silicon MPS GPU detected - training will use GPU!')
elif torch.cuda.is_available():
    print('NVIDIA CUDA GPU detected.')
else:
    print('CPU mode - no GPU found.')
print('Environment OK!')

## Step 2: Start MLflow Server
> Auto-detects a free port (tries 5000, falls back to 5050 on macOS).
> Runs as a background process for the full session.
> Has an auto-restart watchdog. MLflow is ALWAYS alive.
> Re-running this cell safely restarts it if needed.

In [ ]:
# CELL 3 - Start MLflow tracking server (background, always-on)
import subprocess, time, os, socket, mlflow, webbrowser
import requests as req

PROJECT_DIR  = os.path.abspath('')
DB_PATH      = os.path.join(PROJECT_DIR, 'mlflow.db')
ARTIFACT_DIR = os.path.join(PROJECT_DIR, 'mlruns')
os.makedirs(ARTIFACT_DIR, exist_ok=True)

# ── Auto-detect free port (5000 on Linux/Windows, 5050 on macOS if AirPlay blocks 5000) ──
def _port_free(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', port)) != 0

MLFLOW_PORT = 5000 if _port_free(5000) else 5050
print(f'Selected MLflow port: {MLFLOW_PORT}')

# Kill any stale server
os.system("pkill -f 'mlflow server' 2>/dev/null")
time.sleep(2)

# Start MLflow server in background
log_f = open(os.path.join(PROJECT_DIR, 'mlflow.log'), 'w')
mlflow_proc = subprocess.Popen([
    'mlflow', 'server',
    '--backend-store-uri', f'sqlite:///{DB_PATH}',
    '--default-artifact-root', ARTIFACT_DIR,
    '--host', '0.0.0.0',
    '--port', str(MLFLOW_PORT),
    '--allowed-hosts', '*'   # bypasses DNS-rebinding protection
], stdout=log_f, stderr=log_f)

print('Waiting for MLflow to start...')
for i in range(15):
    time.sleep(2)
    try:
        if req.get(f'http://localhost:{MLFLOW_PORT}/health', timeout=2).status_code == 200:
            print(f'MLflow is UP (attempt {i+1})')
            break
    except:
        print(f'  attempt {i+1}/15...')
else:
    print('Server may still be starting. Check {PROJECT_DIR}/mlflow.log')

mlflow.set_tracking_uri(f'sqlite:///{DB_PATH}')
mlflow.set_experiment('CIFAR10_Production_Pipeline')

webbrowser.open(f'http://localhost:{MLFLOW_PORT}')
print(f'MLflow Dashboard -> http://localhost:{MLFLOW_PORT}  (opened in browser)')

## Step 2b: Start MLflow Watchdog (keeps server always alive)
> Run this right after Cell 3. It silently restarts MLflow every 30s if it crashes.

In [ ]:
# CELL 4 - MLflow watchdog (auto-restart if server ever goes down)
import threading, time, subprocess, os
import requests as req

_WATCHDOG_ACTIVE = True

def _restart_mlflow():
    PROJECT_DIR  = os.path.abspath('')
    DB_PATH      = os.path.join(PROJECT_DIR, 'mlflow.db')
    ARTIFACT_DIR = os.path.join(PROJECT_DIR, 'mlruns')
    os.system("pkill -f 'mlflow server' 2>/dev/null")
    time.sleep(2)
    log_f = open(os.path.join(PROJECT_DIR,'mlflow.log'),'a')
    subprocess.Popen([
        'mlflow','server',
        '--backend-store-uri', f'sqlite:///{DB_PATH}',
        '--default-artifact-root', ARTIFACT_DIR,
        '--host','0.0.0.0','--port',str(MLFLOW_PORT),'--allowed-hosts','*'
    ], stdout=log_f, stderr=log_f)
    time.sleep(5)
    print(f'MLflow auto-restarted -> http://localhost:{MLFLOW_PORT}')

def _watchdog():
    while _WATCHDOG_ACTIVE:
        try:
            if req.get(f'http://localhost:{MLFLOW_PORT}/health', timeout=3).status_code != 200:
                raise Exception('bad status')
        except:
            print('MLflow down - auto-restarting...')
            _restart_mlflow()
        time.sleep(30)

threading.Thread(target=_watchdog, daemon=True).start()
print('Watchdog active - checks MLflow every 30 seconds.')
print('If MLflow crashes it will auto-restart without any action from you.')

## Step 3: Load CIFAR-10 Data
> Downloads ~170 MB to `./data/` inside the notebook folder on first run.

In [ ]:
# CELL 5 - Load CIFAR-10
import torch, torchvision, os
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

DATA_DIR = os.path.join(os.path.abspath(''), 'data')

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

train_dataset = torchvision.datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=train_transform)
val_dataset   = torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=val_transform)

# num_workers=0 is required on Mac to avoid multiprocessing fork issues in Jupyter
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=0)

CIFAR10_CLASSES = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')
print(f'Train: {len(train_dataset):,} | Val: {len(val_dataset):,}')
print(f'Classes: {CIFAR10_CLASSES}')

## Step 4: Define CNN (Module 15 architecture + BatchNorm)

In [ ]:
# CELL 6 - CIFAR10Net with BatchNorm + Dropout
import torch.nn as nn

class CIFAR10Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2,2), nn.Dropout2d(0.2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2,2), nn.Dropout2d(0.3),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64*8*8, 512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 10)
        )
    def forward(self, x):
        return self.classifier(self.features(x).view(x.size(0),-1))

# Auto-select best device
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    print('Apple Silicon MPS GPU - fast training!')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print('NVIDIA CUDA GPU.')
else:
    DEVICE = torch.device('cpu')
    print('CPU mode.')

total_params = sum(p.numel() for p in CIFAR10Net().parameters())
print(f'CIFAR10Net ready | {total_params:,} parameters')

## Step 5: Train + Log to MLflow
> Runs 2 experiments with different learning rates.
> Open MLflow (URL printed in Cell 3) to watch metrics update live during training!
>
> **Model Registry:** Each experiment automatically registers the best model under
> `CIFAR10-Production` in the Model Registry with auto-incrementing version numbers.

In [ ]:
# CELL 7 - Training loop with full MLflow logging (Experiment 1)
import mlflow.pytorch, torch.optim as optim, os

PROJECT_DIR = os.path.abspath('')

def run_experiment(run_name, lr, epochs, optimizer_name='Adam'):
    model     = CIFAR10Net().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params({
            'optimizer': optimizer_name, 'learning_rate': lr,
            'epochs': epochs, 'batch_size': 64,
            'device': str(DEVICE), 'architecture': 'CIFAR10Net_BN', 'dataset': 'CIFAR-10'
        })
        best_acc = 0.0
        for epoch in range(epochs):
            model.train()
            running_loss, c_tr, t_tr = 0.0, 0, 0
            for imgs, labels in train_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()
                out  = model(imgs)
                loss = criterion(out, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item() * imgs.size(0)
                _, pred = torch.max(out, 1)
                t_tr += labels.size(0)
                c_tr += (pred == labels).sum().item()

            model.eval()
            c_val, t_val = 0, 0
            with torch.no_grad():
                for imgs, labels in val_loader:
                    imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                    _, pred = torch.max(model(imgs), 1)
                    t_val += labels.size(0)
                    c_val += (pred == labels).sum().item()

            eloss = running_loss / len(train_loader.dataset)
            tacc  = c_tr / t_tr
            vacc  = c_val / t_val
            scheduler.step()
            mlflow.log_metrics({
                'train_loss': eloss, 'train_accuracy': tacc,
                'val_accuracy': vacc, 'learning_rate': scheduler.get_last_lr()[0]
            }, step=epoch)
            if vacc > best_acc:
                best_acc = vacc
                torch.save(model.state_dict(), os.path.join(PROJECT_DIR, f'best_{run_name}.pt'))
            print(f'[{run_name}] Epoch {epoch+1}/{epochs} | Loss:{eloss:.4f} | Train:{tacc:.2%} | Val:{vacc:.2%}')

        mlflow.log_metric('best_val_accuracy', best_acc)
        mlflow.set_tag('best_model_file', f'best_{run_name}.pt')

        # Register the model in MLflow Model Registry
        # This creates/updates 'CIFAR10-Production' with a new version each run
        mlflow.pytorch.log_model(
            model,
            artifact_path='model',
            registered_model_name='CIFAR10-Production'  # ← auto-registers in Model Registry
        )
        print(f'[{run_name}] Done! Best Val: {best_acc:.2%}')
        print(f'[{run_name}] Model registered as CIFAR10-Production (new version created)')
    return model, best_acc

model_a, acc_a = run_experiment('Adam_lr0.001', lr=0.001, epochs=5)

In [ ]:
# CELL 8 - Experiment 2 (different LR, compare in MLflow)
model_b, acc_b = run_experiment('Adam_lr0.003', lr=0.003, epochs=5)
print(f'Comparison -> Exp1: {acc_a:.2%}  |  Exp2: {acc_b:.2%}')

import webbrowser
webbrowser.open(f'http://localhost:{MLFLOW_PORT}')
print(f'\nMLflow Dashboard -> http://localhost:{MLFLOW_PORT}')
print()
print('=' * 65)
print('HOW TO COMPARE RUNS SIDE-BY-SIDE IN MLFLOW')
print('=' * 65)
print('1. Open the MLflow URL above in your browser')
print('2. Click the experiment "CIFAR10_Production_Pipeline" on the left')
print('3. You will see both runs listed (Adam_lr0.001 and Adam_lr0.003)')
print('4. Tick the checkbox next to BOTH runs to select them')
print('5. Click the "Compare" button that appears at the top')
print('6. You will see:')
print('   • Parameters table  - compare LR, optimizer, etc.')
print('   • Metrics table     - compare best_val_accuracy side-by-side')
print('   • Charts            - overlaid train_loss & val_accuracy curves')
print('=' * 65)

## Step 5b: Model Registry - View Registered Versions
> Every `run_experiment()` call automatically registers the trained model under
> **CIFAR10-Production** in the MLflow Model Registry. Each run creates a new version.
>
> **Why a Model Registry?**
> - **Versioning:** v1, v2, v3... — compare any version against any other
> - **Stage Management:** Tag versions as `Staging` → `Production` → `Archived`
> - **Lineage:** Each version links back to its exact training run, parameters, and metrics
> - **Single Source of Truth:** Instead of remembering which run was best, just ask: *"What's the Production version?"*

In [ ]:
# CELL 8b - Query the Model Registry
from mlflow.tracking import MlflowClient

client = MlflowClient(tracking_uri=f'sqlite:///{DB_PATH}')

print('=' * 70)
print('MODEL REGISTRY: CIFAR10-Production')
print('=' * 70)

try:
    # Get all versions of the registered model
    versions = client.search_model_versions("name='CIFAR10-Production'")
    print(f"{'Version':<10} {'Run Name':<25} {'Status':<15} {'Stage'}")
    print('-' * 70)
    for v in versions:
        run = client.get_run(v.run_id)
        run_name = run.info.run_name
        print(f"v{v.version:<9} {run_name:<25} {v.status:<15} {v.current_stage}")
    print()
    print(f'Total registered versions: {len(versions)}')
    print(f'View in UI -> http://localhost:{MLFLOW_PORT}/#/models/CIFAR10-Production')
except Exception as e:
    print(f'Registry query failed: {e}')
    print('This is expected if no experiments have been run yet.')

## Step 6: MLflow Run Summary

In [ ]:
# CELL 9 - Print experiment summary table
from mlflow.tracking import MlflowClient
import os

DB_PATH = os.path.join(os.path.abspath(''), 'mlflow.db')
client  = MlflowClient(tracking_uri=f'sqlite:///{DB_PATH}')
exp     = client.get_experiment_by_name('CIFAR10_Production_Pipeline')
runs    = client.search_runs([exp.experiment_id], order_by=['metrics.best_val_accuracy DESC'])

print(f"{'Run Name':<25} {'LR':<8} {'Epochs':<8} {'Best Val Acc':<15} {'Optimizer'}")
print('-' * 70)
for r in runs:
    p, m = r.data.params, r.data.metrics
    print(f"{r.info.run_name:<25} {p.get('learning_rate','?'):<8} "
          f"{p.get('epochs','?'):<8} {m.get('best_val_accuracy',0):.4f}          "
          f"{p.get('optimizer','?')}")
best = runs[0]
print(f"Best Run: {best.info.run_name} | Val Acc: {best.data.metrics.get('best_val_accuracy',0):.2%}")

## Step 7: Deploy FastAPI Inference Server (http://localhost:8000)
> No ngrok needed - everything is local. Swagger UI opens in your browser automatically.

In [ ]:
# CELL 10 - FastAPI inference server on localhost:8000
import nest_asyncio, threading, uvicorn, json, time, webbrowser, os, torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import requests as req

nest_asyncio.apply()

PROJECT_DIR = os.path.abspath('')
best_model  = CIFAR10Net()
best_model.load_state_dict(
    torch.load(os.path.join(PROJECT_DIR, 'best_Adam_lr0.001.pt'), map_location='cpu')
)
best_model.eval()
print('Model loaded for inference.')

api_app = FastAPI(
    title='CIFAR-10 Inference API',
    description='Module 16 - MLOps Production Demo | Nethaji Nirmal',
    version='1.0.0'
)

class ImagePayload(BaseModel):
    flat_image: list  # 3072 floats - flattened 3x32x32 image

@api_app.get('/')
def root(): return {'message': 'CIFAR-10 MLOps API', 'classes': list(CIFAR10_CLASSES)}

@api_app.get('/health')
def health(): return {'status': 'healthy', 'model': 'CIFAR10Net_BN'}

@api_app.post('/predict')
def predict(payload: ImagePayload):
    if len(payload.flat_image) != 3072:
        raise HTTPException(400, f'Expected 3072 values, got {len(payload.flat_image)}')
    x = torch.tensor(payload.flat_image, dtype=torch.float32).view(1,3,32,32)
    with torch.no_grad():
        probs = torch.softmax(best_model(x), dim=1)[0]
    top3 = torch.topk(probs, 3)
    return {
        'top_prediction': CIFAR10_CLASSES[top3.indices[0].item()],
        'confidence':     round(top3.values[0].item(), 4),
        'top3': [
            {'class': CIFAR10_CLASSES[i.item()], 'prob': round(v.item(),4)}
            for v,i in zip(top3.values, top3.indices)
        ]
    }

threading.Thread(
    target=lambda: uvicorn.run(api_app, host='127.0.0.1', port=8000, log_level='warning'),
    daemon=True
).start()
time.sleep(3)

try:
    r = req.get('http://localhost:8000/health', timeout=3)
    print(f'FastAPI UP: {r.json()}')
except Exception as e:
    print(f'FastAPI check: {e}')

webbrowser.open('http://localhost:8000/docs')
print('FastAPI running:')
print('  Docs    -> http://localhost:8000/docs')
print('  Health  -> http://localhost:8000/health')
print('  Predict -> POST http://localhost:8000/predict')

## Step 8: Client Test - End-to-End Verification

In [ ]:
# CELL 11 - Client test (calls local API)
import requests, json

test_image, true_label_idx = val_dataset[500]
payload = {'flat_image': test_image.numpy().flatten().tolist()}
print(f'Ground Truth: {CIFAR10_CLASSES[true_label_idx]}')

response = requests.post('http://localhost:8000/predict', json=payload)
result   = response.json()
print('--- API Response ---')
print(json.dumps(result, indent=2))
correct = result['top_prediction'] == CIFAR10_CLASSES[true_label_idx]
print(f"{'CORRECT!' if correct else 'Wrong prediction'}")

## Step 9: Shutdown (run when done)

In [1]:
# CELL 12 - Stop all background services
import os
_WATCHDOG_ACTIVE = False
os.system("pkill -f 'mlflow server' 2>/dev/null")
print('MLflow stopped.')
print('FastAPI stops when Jupyter kernel shuts down.')

MLflow stopped.
FastAPI stops when Jupyter kernel shuts down.


## Step 10: Production Docker Concept

```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY app.py best_Adam_lr0.001.pt ./
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

```bash
docker build -t cifar10-api .
docker run -p 8000:8000 cifar10-api
```

**Full MLOps Cycle:**
```
Train -> MLflow -> Register Model -> Best Model -> FastAPI -> Docker -> CI/CD -> Monitor -> Retrain
```

---
## Module 16 Complete - What You Built Locally

| Component | Tool | URL |
|---|---|---|
| Experiment Tracking | MLflow | http://localhost:`MLFLOW_PORT` (auto-detected) |
| **Model Registry** | **mlflow.pytorch.log_model + registered_model_name** | **MLflow UI → Models tab** |
| Live Metrics Charts | mlflow.log_metrics | MLflow UI → Charts |
| Model Artifacts | mlflow.pytorch.log_model | MLflow UI → Artifacts |
| REST API | FastAPI + uvicorn | http://localhost:8000/docs |
| Live Inference | POST /predict | Cell 11 output |
| Auto-Restart Guard | Python watchdog thread | Cell 4 (always active) |
| Containerization | Docker | Concept (Cell 12) |
